# Approach 1 — Fit on averaged raw CE

**Pipeline:** Average raw CE_TEST across 100 runs at each batch number → fit one curve `CE(x) = A + B/(x+1)^n` → compute IPA from that single fit.

**Input:** `prune_layers_ALL/p-percentage_{p}/batch_size_{bs}/averaged_runs_p_{p}_bs_{bs}.csv` (column `Avg_CE_Test`).

**Output:** `intermediate/approach_1_fit_params_bs_{bs}.csv` with every `P%, A, B, n, CE_o, CE_L, learn_BN, IPA` row, plus `ipa_summary_approach_1.csv`.

**IPA:** `abs(CE_o - CE_L) / learn_BN` where `CE_L = CE_o - 0.9*(CE_o - A)`.

In [1]:
# === Cell 1 — Config, imports, helpers ===
import os, glob, re
import numpy as np
import pandas as pd
from lmfit import Parameters, minimize
import warnings
warnings.filterwarnings("ignore")

# Paths
BASE_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
FIT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\Fitting_IPA_curves_data_I"
OUT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1"
INTERMEDIATE_DIR = os.path.join(OUT_DIR, "intermediate")
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

BATCH_SIZES = [64, 1024, 60000]
CE_o = np.log(10)   # max CE for 10-class problem, ~2.302585

# Auto-detect pruning percentages from prune_layers_ALL/p-percentage_*/
p_dirs = glob.glob(os.path.join(BASE_DIR, "p-percentage_*"))
PRUNING_LEVELS = sorted([
    float(re.search(r"p-percentage_([\d.]+)", d).group(1))
    for d in p_dirs
])
print(f"Found {len(PRUNING_LEVELS)} pruning percentages: {PRUNING_LEVELS}")
print(f"CE_o = ln(10) = {CE_o:.6f}")

# Fit-function helpers (verbatim from fitting_function_IPA.ipynb)
A_MIN, A_MAX = 0.1, 2.3
B_MIN, B_MAX = 0, 1000
N_MIN, N_MAX = 0.5, 3.0


def initialize_guesses(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    A0 = np.percentile(y, 5)
    B0 = np.percentile(y, 95) - A0
    n0 = 0.5
    if len(x) > 10:
        denom = y[0] - A0
        if abs(denom) > 1e-10:
            frac = max(1e-6, (y[0] - y[-1]) / denom)
            if frac > 0:
                n0 = max(0.3, min(1.5, -np.log(frac)))
    return A0, n0, B0


def model(params, x):
    vals = params.valuesdict()
    A, B, n = vals["A"], vals["B"], vals["n"]
    return A + B / ((x + 1) ** n)


def residual(params, x, data):
    weight = x
    return weight * (model(params, x) - data)


def fit_curve(x, y):
    mask  = ~np.isnan(y)
    x_fit = np.asarray(x)[mask]
    y_fit = np.asarray(y)[mask]
    if len(x_fit) < 10:
        return None
    A0, n0, B0 = initialize_guesses(x_fit, y_fit)
    params = Parameters()
    params.add("A", value=A0, min=A_MIN, max=A_MAX)
    params.add("B", value=B0, min=B_MIN, max=B_MAX)
    params.add("n", value=n0, min=N_MIN, max=N_MAX)
    try:
        return minimize(residual, params, args=(x_fit, y_fit))
    except Exception:
        return None

# --- IPA from fit (single source of truth) ---
# CE_L is the physically meaningful learning threshold.
#   CE_L = CE_o - 0.9 * (CE_o - A)
#   IPA  = abs(CE_o - CE_L) / learn_BN   where learn_BN = first BN in x_grid with fitted CE <= CE_L.
# We intentionally do NOT simplify to 0.9*(CE_o - A)/learn_BN — CE_L stays a first-class variable.
def compute_ipa_from_fit(x_grid, A, B, n):
    x_grid = np.asarray(x_grid, dtype=float)
    CE_L = CE_o - 0.9 * (CE_o - A)
    fitted = A + B / ((x_grid + 1) ** n)
    mask = fitted <= CE_L

    if mask.any():
        # In-range: original discrete-search behavior
        learn_BN = float(x_grid[mask][0])
        fitted_at = float(fitted[mask][0])
    else:
        # Out-of-range: extrapolate analytically, then ceil to integer BN
        denom = CE_L - A
        if denom <= 0 or n <= 0 or B <= 0:
            return {"CE_L": CE_L, "learn_BN": np.nan, "IPA": np.nan, "fitted_at_learn_BN": np.nan}
        BN_analytic = (B / denom) ** (1.0 / n) - 1.0
        if not np.isfinite(BN_analytic) or BN_analytic <= 0:
            return {"CE_L": CE_L, "learn_BN": np.nan, "IPA": np.nan, "fitted_at_learn_BN": np.nan}
        learn_BN = float(np.ceil(BN_analytic))
        fitted_at = float(A + B / ((learn_BN + 1) ** n))

    if learn_BN == 0:
        return {"CE_L": CE_L, "learn_BN": 0.0, "IPA": np.nan, "fitted_at_learn_BN": fitted_at}
    IPA = abs(CE_o - CE_L) / learn_BN
    return {"CE_L": CE_L, "learn_BN": learn_BN, "IPA": IPA, "fitted_at_learn_BN": fitted_at}
print("Cell 1 ready.")


Found 19 pruning percentages: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]
CE_o = ln(10) = 2.302585
Cell 1 ready.


In [2]:
# === Cell 2 — Approach 1: fit averaged raw CE, then IPA from fit ===
# Per (P%, BS): load averaged_runs_p_{p}_bs_{bs}.csv, fit Avg_CE_Test vs Batch_Number,
# call compute_ipa_from_fit with the file's BN grid.
inter_by_bs = {}

for bs in BATCH_SIZES:
    print("\n" + "=" * 70)
    print(f"  Approach 1 — Batch size {bs}")
    print("=" * 70)
    rows = []
    for p in PRUNING_LEVELS:
        avg_csv = os.path.join(BASE_DIR, f"p-percentage_{p}", f"batch_size_{bs}",
                               f"averaged_runs_p_{p}_bs_{bs}.csv")
        if not os.path.exists(avg_csv):
            print(f"  [SKIP] P%={p*100:5.1f}%  — missing {avg_csv}")
            continue
        df = pd.read_csv(avg_csv)
        df.columns = df.columns.str.strip()
        ce_col = next((c for c in df.columns if c in ("Avg_CE_Test", "Avg_CE_test")), None)
        bn_col = next((c for c in df.columns if "Batch" in c), None)
        if ce_col is None or bn_col is None:
            print(f"  [SKIP] P%={p*100:5.1f}%  — unexpected columns {list(df.columns)}")
            continue
        df = df.dropna(subset=[ce_col, bn_col])
        x = df[bn_col].values.astype(float)
        y = df[ce_col].values.astype(float)

        result = fit_curve(x, y)
        if result is None:
            print(f"  [FAIL] P%={p*100:5.1f}%  — fit did not converge")
            continue
        A = result.params["A"].value
        B = result.params["B"].value
        n = result.params["n"].value
        ipa = compute_ipa_from_fit(x, A, B, n)

        print(f"  P%={p*100:5.1f}%  A={A:.4f}  B={B:.4f}  n={n:.4f}  "
              f"CE_o={CE_o:.4f}  CE_L={ipa['CE_L']:.4f}  learn_BN={ipa['learn_BN']!r:>8}  IPA={ipa['IPA']}")

        rows.append({
            "P%": p * 100, "A": A, "B": B, "n": n,
            "CE_o": CE_o, "CE_L": ipa["CE_L"],
            "learn_BN": ipa["learn_BN"], "fitted_at_learn_BN": ipa["fitted_at_learn_BN"],
            "IPA": ipa["IPA"],
        })

    if rows:
        bs_df = pd.DataFrame(rows)
        inter_path = os.path.join(INTERMEDIATE_DIR, f"approach_1_fit_params_bs_{bs}.csv")
        bs_df.to_csv(inter_path, index=False)
        print(f"  Saved: {inter_path}")
        inter_by_bs[bs] = bs_df

print("\n[Cell 2 done]")



  Approach 1 — Batch size 64
  P%=  0.0%  A=0.2993  B=5.8068  n=0.9046  CE_o=2.3026  CE_L=0.4997  learn_BN=    41.0  IPA=0.04397381330460307
  P%= 10.0%  A=0.2927  B=5.7789  n=0.8771  CE_o=2.3026  CE_L=0.4937  learn_BN=    46.0  IPA=0.03932322281468485
  P%= 20.0%  A=0.2986  B=7.2259  n=0.9292  CE_o=2.3026  CE_L=0.4990  learn_BN=    47.0  IPA=0.038373407658194074
  P%= 30.0%  A=0.2993  B=7.6436  n=0.9203  CE_o=2.3026  CE_L=0.4997  learn_BN=    52.0  IPA=0.034671608378540344
  P%= 40.0%  A=0.2986  B=8.4591  n=0.9154  CE_o=2.3026  CE_L=0.4990  learn_BN=    59.0  IPA=0.030569272154594544
  P%= 50.0%  A=0.2828  B=7.1199  n=0.8119  CE_o=2.3026  CE_L=0.4847  learn_BN=    80.0  IPA=0.022722981546372638
  P%= 60.0%  A=0.2847  B=7.5459  n=0.7812  CE_o=2.3026  CE_L=0.4865  learn_BN=   103.0  IPA=0.017631850300328926


  P%= 70.0%  A=0.3018  B=8.6419  n=0.7678  CE_o=2.3026  CE_L=0.5019  learn_BN=   134.0  IPA=0.013437937264331169


  P%= 80.0%  A=0.3154  B=8.2479  n=0.6761  CE_o=2.3026  CE_L=0.5141  learn_BN=   247.0  IPA=0.007240838320295698
  P%= 82.0%  A=0.3653  B=10.1706  n=0.7322  CE_o=2.3026  CE_L=0.5590  learn_BN=   223.0  IPA=0.007818792403666312
  P%= 84.0%  A=0.3550  B=9.0167  n=0.6703  CE_o=2.3026  CE_L=0.5498  learn_BN=   305.0  IPA=0.005746843489890376
  P%= 86.0%  A=0.5373  B=0.9990  n=0.5000  CE_o=2.3026  CE_L=0.7138  learn_BN=    32.0  IPA=0.04964946690117183
  P%= 88.0%  A=0.4376  B=9.9780  n=0.6712  CE_o=2.3026  CE_L=0.6241  learn_BN=   375.0  IPA=0.0044760729690091535
  P%= 90.0%  A=0.4543  B=8.8413  n=0.6096  CE_o=2.3026  CE_L=0.6392  learn_BN=   569.0  IPA=0.0029234159990313008
  P%= 92.0%  A=0.4552  B=6.8571  n=0.5075  CE_o=2.3026  CE_L=0.6399  learn_BN=  1238.0  IPA=0.0013430033735678012
  P%= 94.0%  A=0.5428  B=7.5046  n=0.5000  CE_o=2.3026  CE_L=0.7188  learn_BN=  1818.0  IPA=0.000871162979903173
  P%= 96.0%  A=0.7119  B=8.2064  n=0.5000  CE_o=2.3026  CE_L=0.8709  learn_BN=  2660.0  IPA=0

  P%= 98.0%  A=1.5180  B=0.6803  n=0.5000  CE_o=2.3026  CE_L=1.5965  learn_BN=    75.0  IPA=0.009414931115928553


  P%=100.0%  A=2.3000  B=0.0000  n=0.5000  CE_o=2.3026  CE_L=2.3003  learn_BN=     0.0  IPA=nan
  Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\intermediate\approach_1_fit_params_bs_64.csv

  Approach 1 — Batch size 1024
  P%=  0.0%  A=0.2738  B=6.0539  n=1.0175  CE_o=2.3026  CE_L=0.4767  learn_BN=    28.0  IPA=0.06521004187305483
  P%= 10.0%  A=0.2738  B=7.7385  n=1.0722  CE_o=2.3026  CE_L=0.4767  learn_BN=    29.0  IPA=0.0629613933331071
  P%= 20.0%  A=0.2630  B=6.4888  n=0.9769  CE_o=2.3026  CE_L=0.4670  learn_BN=    34.0  IPA=0.05398922104508974
  P%= 30.0%  A=0.2650  B=7.4580  n=0.9958  CE_o=2.3026  CE_L=0.4688  learn_BN=    37.0  IPA=0.04956297150340743
  P%= 40.0%  A=0.2578  B=6.8886  n=0.9278  CE_o=2.3026  CE_L=0.4622  learn_BN=    44.0  IPA=0.04182592971809077
  P%= 50.0%  A=0.2540  B=6.8181  n=0.8783  CE_o=2.3026  CE_L=0.4588  learn_BN=    54.0  IPA=0.03414356611545608
  P%= 60.0%  A=0.2642  B=7.9399  n=0.8799  CE_o=2.3026  CE_

  P%= 82.0%  A=0.3700  B=18.8979  n=0.9662  CE_o=2.3026  CE_L=0.5633  learn_BN=   114.0  IPA=0.01525694298951248
  P%= 84.0%  A=0.3228  B=8.2877  n=0.7021  CE_o=2.3026  CE_L=0.5207  learn_BN=   204.0  IPA=0.008734562844850365
  P%= 86.0%  A=0.4733  B=0.9669  n=0.5000  CE_o=2.3026  CE_L=0.6562  learn_BN=    27.0  IPA=0.06097650309980153
  P%= 88.0%  A=0.3960  B=8.8796  n=0.6866  CE_o=2.3026  CE_L=0.5867  learn_BN=   268.0  IPA=0.006402708728467529
  P%= 90.0%  A=0.5764  B=1.0400  n=0.5000  CE_o=2.3026  CE_L=0.7490  learn_BN=    36.0  IPA=0.04315462732485115
  P%= 92.0%  A=0.4490  B=7.0494  n=0.5638  CE_o=2.3026  CE_L=0.6344  learn_BN=   635.0  IPA=0.0026271057460142398
  P%= 94.0%  A=0.8369  B=1.0307  n=0.5000  CE_o=2.3026  CE_L=0.9834  learn_BN=    49.0  IPA=0.026921389463155948
  P%= 96.0%  A=1.0783  B=0.9382  n=0.5000  CE_o=2.3026  CE_L=1.2007  learn_BN=    58.0  IPA=0.018997708339562783
  P%= 98.0%  A=1.4415  B=0.7284  n=0.5000  CE_o=2.3026  CE_L=1.5276  learn_BN=    71.0  IPA=0.010

  Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\intermediate\approach_1_fit_params_bs_1024.csv

  Approach 1 — Batch size 60000
  P%=  0.0%  A=0.3279  B=10.9286  n=1.3979  CE_o=2.3026  CE_L=0.5253  learn_BN=    17.0  IPA=0.1045439184391034
  P%= 10.0%  A=0.2983  B=7.8895  n=1.1313  CE_o=2.3026  CE_L=0.4988  learn_BN=    25.0  IPA=0.07215278950062076
  P%= 20.0%  A=0.2790  B=6.5342  n=1.0111  CE_o=2.3026  CE_L=0.4813  learn_BN=    31.0  IPA=0.05874976184966372
  P%= 30.0%  A=0.2707  B=6.6183  n=0.9753  CE_o=2.3026  CE_L=0.4739  learn_BN=    35.0  IPA=0.05224955216692205
  P%= 40.0%  A=0.2655  B=6.9377  n=0.9511  CE_o=2.3026  CE_L=0.4692  learn_BN=    40.0  IPA=0.04583410145754196
  P%= 50.0%  A=0.2563  B=6.7776  n=0.8949  CE_o=2.3026  CE_L=0.4609  learn_BN=    49.0  IPA=0.037584865255140765
  P%= 60.0%  A=0.2629  B=7.9060  n=0.8954  CE_o=2.3026  CE_L=0.4669  learn_BN=    59.0  IPA=0.031113692117494512
  P%= 70.0%  A=0.2894  B=10.6143  n=0

  P%= 80.0%  A=0.2949  B=8.9502  n=0.7828  CE_o=2.3026  CE_L=0.4956  learn_BN=   127.0  IPA=0.014227916498376948
  P%= 82.0%  A=0.3287  B=10.4481  n=0.8210  CE_o=2.3026  CE_L=0.5261  learn_BN=   125.0  IPA=0.014211672135317377
  P%= 84.0%  A=0.3352  B=9.5130  n=0.7617  CE_o=2.3026  CE_L=0.5319  learn_BN=   162.0  IPA=0.010929923503463754
  P%= 86.0%  A=0.3610  B=10.0001  n=0.7607  CE_o=2.3026  CE_L=0.5551  learn_BN=   177.0  IPA=0.009872647709417851
  P%= 88.0%  A=0.3589  B=8.3775  n=0.6738  CE_o=2.3026  CE_L=0.5533  learn_BN=   266.0  IPA=0.006576258814328385
  P%= 90.0%  A=0.4044  B=8.2350  n=0.6474  CE_o=2.3026  CE_L=0.5942  learn_BN=   338.0  IPA=0.005054302389134038
  P%= 92.0%  A=0.4277  B=6.7388  n=0.5504  CE_o=2.3026  CE_L=0.6152  learn_BN=   670.0  IPA=0.0025185434757166872
  P%= 94.0%  A=0.8349  B=1.0307  n=0.5000  CE_o=2.3026  CE_L=0.9816  learn_BN=    49.0  IPA=0.026958261912135538
  P%= 96.0%  A=1.0226  B=0.9713  n=0.5000  CE_o=2.3026  CE_L=1.1506  learn_BN=    57.0  IPA=0

In [3]:
# === Cell 3 — Build wide summary CSV for Approach 1 ===
# Schema: P%, IPA_Avg_64, STD_64, IPA_Avg_1024, STD_1024, IPA_Avg_60000, STD_60000
# STD columns blank (NaN) for single-curve approaches; populated only by 2b.
summary_rows = []
for p in PRUNING_LEVELS:
    row = {"P%": p * 100}
    for bs in BATCH_SIZES:
        df = inter_by_bs.get(bs)
        if df is None:
            mean_val, std_val = np.nan, np.nan
        else:
            sub = df[df["P%"] == p * 100]
            mean_val = float(sub["IPA"].iloc[0]) if (not sub.empty and "IPA" in sub.columns) else (
                       float(sub["IPA_mean"].iloc[0]) if (not sub.empty and "IPA_mean" in sub.columns) else np.nan)
        std_val  = np.nan
        row[f"IPA_Avg_{bs}"] = mean_val
        row[f"STD_{bs}"]     = std_val
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows, columns=[
    "P%", "IPA_Avg_64", "STD_64", "IPA_Avg_1024", "STD_1024", "IPA_Avg_60000", "STD_60000"
])
out_csv = os.path.join(OUT_DIR, "ipa_summary_approach_1.csv")
summary_df.to_csv(out_csv, index=False)
print(f"\nFinal summary written: {out_csv}")
print(summary_df.to_string(index=False))



Final summary written: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\ipa_summary_approach_1.csv
   P%  IPA_Avg_64  STD_64  IPA_Avg_1024  STD_1024  IPA_Avg_60000  STD_60000
  0.0    0.043974     NaN      0.065210       NaN       0.104544        NaN
 10.0    0.039323     NaN      0.062961       NaN       0.072153        NaN
 20.0    0.038373     NaN      0.053989       NaN       0.058750        NaN
 30.0    0.034672     NaN      0.049563       NaN       0.052250        NaN
 40.0    0.030569     NaN      0.041826       NaN       0.045834        NaN
 50.0    0.022723     NaN      0.034144       NaN       0.037585        NaN
 60.0    0.017632     NaN      0.028665       NaN       0.031114        NaN
 70.0    0.013438     NaN      0.020526       NaN       0.026646        NaN
 80.0    0.007241     NaN      0.012833       NaN       0.014228        NaN
 82.0    0.007819     NaN      0.015257       NaN       0.014212        NaN
 84.0    0.005747     NaN